# Outlook

In this notebook, using BBRL, you will study some effects of partial observability
on the Swimmer-v5 environment, using the TD3 algorithm.

To emulate partial observability, you will design dedicated wrappers. Then you will study
whether extending the input of the agent policy and critic with a memory of previous states
and can help solve the partial observability issue. Yu will also study whether using action chunks
instead of single actions as ouptput has an effect of the learning performance.
This will also be achieved by designing other temporal extension wrappers.

# Installation

In [89]:
# Prepare the environment

import os
import copy
import numpy as np
import gymnasium as gym
import math
import bbrl_gymnasium  # noqa: F401
import torch
import torch.nn as nn
from bbrl.agents import Agent, Agents, TemporalAgent
from bbrl_utils.algorithms import EpochBasedAlgo
from bbrl_utils.nn import build_mlp, setup_optimizer, soft_update_params
from bbrl_utils.notebook import setup_tensorboard
from bbrl.visu.plot_policies import plot_policy
from omegaconf import OmegaConf

import bbrl_utils

bbrl_utils.setup()

In [2]:
print(bbrl_utils.__file__)

C:\Users\tomto\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\bbrl_utils\__init__.py


# Temporal modification wrappers

The [Swimmer-v5](https://gymnasium.farama.org/environments/mujoco/swimmer/) environment is a gymnasium environment.
See the gymnasium page for a description of the state and action spaces.

To emulate partial observability in Swimmer-v5, you will hide either the $qpos$ or the $qvel$ features, or both,
by filtering them out of the state returned by the environment.
This is implemented with the ```FeatureFilterWrapper```.

To compensate for partial observability, you will extend the architecture of the agent
with a memory of previous states and extend its output with action chunks.
This is implemented with two wrappers, the ```ObsTimeExtensionWrapper```
and the ```ActionTimeExtensionWrapper```.

## The FeatureFilterWrapper

The FeatureFilterWrapper removes a feature from the returned observation
when calling the ```reset()``` and ```step(action)``` functions.
The index of the removed feature is given as a parameter when building the object.

To filter out the $qpos$ or $qvel$ features from the Swimmer-v5 environment,
the idea is to call the wrapper the adequate number of times, using something like
```env = FeatureFilterWrapper(FeatureFilterWrapper(inner_env, X), Y)```
where ```inner_env``` is the Swimmer-v5 environment
and X and Y are position of features you want to filter out.

Beware of filtering features in the right order,
as removing a feature changes the index of all subsequent features.

One way to put such a wrapper with a parameter into the list of wrappers is to use a lambda:
for instance, `lambda env: FeatureFilterWrapper(env, 3),`

### Exercise 1: code the FeatureFilterWrapper class below.

Beyond rewriting the ```reset()``` and ```step(action)``` functions,
beware of adapting the observation space and its shape.

In [ ]:
# [[STUDENT]]...

assert False, 'Not implemented yet'


## The ObsTimeExtensionWrapper

When facing a partially observable environment, training with RL a reactive agent which just selects an action based on the current observation
is not guaranteed to reach optimality. An option to mitigate this fundamental limitation is to equip the agent with a memory of the past.

One way to do so is to use a recurrent neural network instead of a feedforward one to implement the agent: the neural network contains
some memory capacity and the RL process may tune this internal memory so as to remember exactly what is necessary from the
past observation. This has been done many times using an LSTM, see for instance
[this early paper](https://proceedings.neurips.cc/paper/2001/file/a38b16173474ba8b1a95bcbc30d3b8a5-Paper.pdf).

Another way to do so is to equip the agent with a list-like memory of the past observations
and to extend the critic and policy to take as input the current observation and the previous ones.
This removes the difficulty of learning an adequate representation of the past, but this results in
enlarging the input size of the actor and critic networks. This can only be done if the required memory
horizon to behave optimally is small enough.

In the case of the Swimmer-v5 environment, one can immediately see that a memory of the previous
observation ($qpos$) is enough to compensate for the absence of the derivative features ($qvel$),
since $\dot{a} \approx (a_{t} - a_{t-1})$.

So we will extend the RL agent with a memory of size 1.

Though it may not be intuitive at first glance, the simplest way to do so is to embed the environment
into a wrapper which contains the required memory and produces the extended observations.
This way, the RL agent will naturally be built with an extended observation space,
and the wrapper will be in charge of concatenating the memorized observation from the previous step
with the current observation received from the inner environment when calling the ```step(action)``` function.
When calling the ```reset()``` function, the memory of observations should be reinitialized with null observations.

### Exercise 2: code the ObsTimeExtensionWrapper class below.

Beyond rewriting the ```reset()``` and ```step(action)``` functions, beware of adapting the observation space and its shape.

In [ ]:
# [[STUDENT]]...

assert False, 'Not implemented yet'


## The ActionTimeExtensionWrapper

It has been observed that, in partially observable environments, preparing to play
a sequence of actions and only playing the first can be better than only preparing for one action.
The difference comes from the fact that the critic evaluates
sequences of actions, even if only the first is played in practice.

Similarly to the ObsTimeExtensionWrapper, the corresponding behavior can be implemented with a wrapper.
The size of the action space of the extended environment should be
M times the size of the action space of the inner environment. This ensures that the policy and the critic
will consider extended actions.
Besides, the ```step(action)``` function should receive an extended actions of size M times
the size of an action, and should only transmit the first action to the inner environment.

Warning, in gymnasium the case where the action is one dimensional requires a slightly
different treatment with respect to when it is multi-dimensional

### Exercise 3: code the ActionTimeExtensionWrapper class below.

Beyond rewriting the ```reset()``` and ```step(action)``` functions, beware of adapting the action space and its shape.

In [ ]:
# [[STUDENT]]...

assert False, 'Not implemented yet'


In [ ]:
class TD3(EpochBasedAlgo):
    def __init__(self, cfg, wrappers_factory):
        super().__init__(cfg, wrappers_factory)

        # Define the agents and optimizers for TD3

        assert False, 'Not implemented yet'





def run_td3(td3: TD3):
    for rb in td3.iter_replay_buffers():
        rb_workspace = rb.get_shuffled(td3.cfg.algorithm.batch_size)

        # Implement the learning loop

        assert False, 'Not implemented yet'


## Launching tensorboard to visualize the results

In [2]:
setup_tensorboard("./outputs")

Launch tensorboard from the shell: 
C:\Users\tomto\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0/tensorboard --logdir 'c:\Users\tomto\Desktop\PROJECT\outputs'


# Experimental study

To run the experiments below, you can use the [TD3](http://proceedings.mlr.press/v80/fujimoto18a/fujimoto18a.pdf) algorithm.

You can just copy paste here the code you have used during the corresponding labs.
We only provide a suggested set of hyper-parameters working well on the Swimmer-v5 environment for TD3.

## Definition of the parameters

The logger is defined as `bbrl.utils.logger.TFLogger` so as to use a
tensorboard visualisation.

In [ ]:
class ContinuousQAgent(Agent):
    def __init__(self, state_dim, hidden_layers, action_dim):
        super().__init__()
        self.is_q_function = True
        self.model = build_mlp(
            [state_dim + action_dim] + list(hidden_layers) + [1], activation=nn.ReLU()
        )

    def forward(self, t):
        # Get the current state $s_t$ and the chosen action $a_t$
        obs = self.get(("env/env_obs", t))
        action = self.get(("action", t))

        # Compute the Q-value(s_t, a_t)
        obs_act = torch.cat((obs, action), dim=1)
        q_value = self.model(obs_act).squeeze(-1)
        self.set((f"{self.prefix}q_value", t), q_value)

In [3]:
class ContinuousDeterministicActor(Agent):
    def __init__(self, state_dim, hidden_layers, action_dim):
        super().__init__()
        layers = [state_dim] + list(hidden_layers) + [action_dim]
        self.model = build_mlp(
            layers, activation=nn.ReLU(), output_activation=nn.Tanh()
        )

    def forward(self, t, **kwargs):
        obs = self.get(("env/env_obs", t))
        action = self.model(obs)
        self.set(("action", t), action)

In [4]:
from torch.distributions import Normal

In [5]:
class AddGaussianNoise(Agent):
    def __init__(self, sigma):
        super().__init__()
        self.sigma = sigma

    def forward(self, t, **kwargs):
        act = self.get(("action", t))
        dist = Normal(act, self.sigma)
        action = dist.sample()
        self.set(("action", t), action)

In [6]:
mse = nn.MSELoss()

def compute_critic_loss(cfg,reward,must_bootstrap,q_values, target_q_values):
    # Compute temporal difference
    q_pred = q_values[0]
    q_t1 = target_q_values[1].detach()

    target = reward[1] + cfg.algorithm.discount_factor * q_t1 * must_bootstrap[1]
    critic_loss = mse(q_pred, target)
    return critic_loss

In [7]:
def compute_actor_loss(q_values):
    return -q_values[0].mean()

In [71]:
class TD3(EpochBasedAlgo):
    def __init__(self, cfg):
        super().__init__(cfg)

        obs_size, act_size = self.train_env.get_obs_and_actions_sizes()

        self.critic_1 = ContinuousQAgent(
            obs_size, cfg.algorithm.architecture.critic_hidden_size, act_size
        ).with_prefix("critic_1/")
        self.target_critic_1 = copy.deepcopy(self.critic_1).with_prefix("target-critic_1/")
        
        self.critic_2 = ContinuousQAgent(
            obs_size, cfg.algorithm.architecture.critic_hidden_size, act_size
        ).with_prefix("critic_2/")
        self.target_critic_2 = copy.deepcopy(self.critic_2).with_prefix("target-critic_2/")

        self.actor = ContinuousDeterministicActor(
            obs_size, cfg.algorithm.architecture.actor_hidden_size, act_size
        )
        self.target_actor = copy.deepcopy(self.actor)

        noise_agent = AddGaussianNoise(cfg.algorithm.action_noise)

        self.train_policy = Agents(self.actor, noise_agent)
        self.eval_policy = self.actor

        for p in self.target_critic_1.parameters(): p.requires_grad = False
        for p in self.target_critic_2.parameters(): p.requires_grad = False
        for p in self.target_actor.parameters():    p.requires_grad = False
        
        self.actor_optimizer = setup_optimizer(cfg.actor_optimizer, self.actor)
        self.critic_optimizer = setup_optimizer(cfg.critic_optimizer, self.critic_1, self.critic_2)

def run_td3(td3: TD3):
    for rb in td3.iter_replay_buffers():
        rb_workspace = rb.get_shuffled(td3.cfg.algorithm.batch_size)

        # compute q <- critic(s,a)
        td3.critic_1(rb_workspace, t=0)
        td3.critic_2(rb_workspace, t=0)
        q1 = rb_workspace["critic_1/q_value"]
        q2 = rb_workspace["critic_2/q_value"]
        reward, terminated = rb_workspace["env/reward", "env/terminated"]

        # a' = target_actor(s') + noise
        td3.target_actor(rb_workspace, t=1)
        target_action = rb_workspace["action"][1] 
        noise = torch.randn_like(target_action) * td3.cfg.algorithm.target_policy_noise
        noise = noise.clamp(
            -td3.cfg.algorithm.target_policy_noise_clip, 
            td3.cfg.algorithm.target_policy_noise_clip
        )
        low = torch.tensor(td3.train_env.action_space.low, device=target_action.device)
        high = torch.tensor(td3.train_env.action_space.high, device=target_action.device)
        smoothed_target_action = torch.max(torch.min(target_action + noise, high), low)

        rb_workspace.set("action", 1,smoothed_target_action)

        # compute t_q <- target_critic(s',a')
        with torch.no_grad():
            td3.target_critic_1(rb_workspace, t=1)
            td3.target_critic_2(rb_workspace, t=1)
            target_q1 = rb_workspace["target-critic_1/q_value"]
            target_q2 = rb_workspace["target-critic_2/q_value"]

        # y = r + gamma * (1 - done) * min(q1_t, q2_t)
        min_q = torch.min(target_q1, target_q2)
        must_bootstrap = ~terminated

        loss_q1 = compute_critic_loss(td3.cfg, reward, must_bootstrap, q1, min_q)
        loss_q2 = compute_critic_loss(td3.cfg, reward, must_bootstrap, q2, min_q)
        td3.logger.add_log("critic_1_loss", loss_q1, td3.nb_steps)
        td3.logger.add_log("critic_2_loss", loss_q2, td3.nb_steps)

        critic_loss = loss_q1 + loss_q2

        #Weights update on critics
        td3.critic_optimizer.zero_grad()
        critic_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            td3.critic_1.parameters(), td3.cfg.algorithm.max_grad_norm
        )

        torch.nn.utils.clip_grad_norm_(
            td3.critic_2.parameters(), td3.cfg.algorithm.max_grad_norm
        )
        td3.critic_optimizer.step()

        # Si step % policy_delay == 0
        if td3.nb_steps % td3.cfg.algorithm.policy_delay == 0:
            # loss_actor = -critic_1(s, actor(s)).mean()
            td3.actor(rb_workspace, t=0)
            td3.critic_1(rb_workspace, t=0)
            q_values = rb_workspace["critic_1/q_value"]
            loss_actor = compute_actor_loss(q_values)

            #update of Actor's weights by backprop on critic
            td3.actor_optimizer.zero_grad()
            loss_actor.backward()
            torch.nn.utils.clip_grad_norm_(
                td3.actor.parameters(), td3.cfg.algorithm.max_grad_norm
            )
            td3.actor_optimizer.step()

            # soft_updates
            soft_update_params(td3.actor, td3.target_actor, td3.cfg.algorithm.tau_target)
            soft_update_params(td3.critic_1, td3.target_critic_1, td3.cfg.algorithm.tau_target)
            soft_update_params(td3.critic_2, td3.target_critic_2, td3.cfg.algorithm.tau_target)

        if td3.evaluate():
            if td3.cfg.plot_agents:
                plot_policy(
                    td3.actor,
                    td3.eval_env,
                    td3.best_reward,
                    str(td3.base_dir / "plots"),
                    td3.cfg.gym_env.env_name,
                    stochastic=False,
                )


In [ ]:
params = {
    "save_best": False,
    "base_dir": "${gym_env.env_name}/td3-S${algorithm.seed}_${current_time:}",
    "collect_stats": False,
    # Set to true to have an insight on the learned policy
    # (but slows down the evaluation a lot!)
    "plot_agents": False,
    "algorithm": {
        "policy_delay" : 2,
        'target_policy_noise': 0.2, #0.2
        "target_policy_noise_clip": 0.5, #0.5
        "seed": 6,
        "max_grad_norm": 0.5,
        "n_envs": 1,
        "n_steps": 1000,
        "nb_evals": 10,#10
        "discount_factor": 0.99999,#0.99999
        "buffer_size": 1e6, #1e6
        "batch_size": 256,
        "tau_target": 0.005,#0.005
        "eval_interval": 5000,#5000
        "max_epochs": 1500,
        # Minimum number of transitions before learning starts
        "learning_starts": 10000,
        "action_noise": 0.1,#0.1
        "architecture": {
            "actor_hidden_size": [400, 300],
            "critic_hidden_size": [400, 300],
        },
    },
    "gym_env": {
        "env_name": "Swimmer-v5",
    },
    "actor_optimizer": {
        "classname": "torch.optim.Adam",
        "lr": 3e-4,#1e-3 3e-4
        "eps": 5e-5,
    },
    "critic_optimizer": {
        "classname": "torch.optim.Adam",
        "lr": 3e-4,
        "eps": 5e-5,
    },
}

In [73]:
td3 = TD3(OmegaConf.create(params))
run_td3(td3)
td3.visualize_best()

  0%|          | 0/3000 [00:00<?, ?it/s]

Video of best agent recorded in folder outputs\Swimmer-v5\td3-S2_20260210-201906\best_agent
moviepy is not installed, skipping video display


In [86]:
class TD3(EpochBasedAlgo):
    def __init__(self, cfg):
        super().__init__(cfg)

        obs_size, act_size = self.train_env.get_obs_and_actions_sizes()

        self.critic_1 = ContinuousQAgent(
            obs_size, cfg.algorithm.architecture.critic_hidden_size, act_size
        ).with_prefix("critic_1/")
        self.target_critic_1 = copy.deepcopy(self.critic_1).with_prefix("target-critic_1/")
        
        self.critic_2 = ContinuousQAgent(
            obs_size, cfg.algorithm.architecture.critic_hidden_size, act_size
        ).with_prefix("critic_2/")
        self.target_critic_2 = copy.deepcopy(self.critic_2).with_prefix("target-critic_2/")

        self.actor = ContinuousDeterministicActor(
            obs_size, cfg.algorithm.architecture.actor_hidden_size, act_size
        )
        self.target_actor = copy.deepcopy(self.actor)

        noise_agent = AddGaussianNoise(cfg.algorithm.action_noise)

        self.train_policy = Agents(self.actor, noise_agent)
        self.eval_policy = self.actor

        for p in self.target_critic_1.parameters(): p.requires_grad = False
        for p in self.target_critic_2.parameters(): p.requires_grad = False
        for p in self.target_actor.parameters():    p.requires_grad = False
        
        self.actor_optimizer = setup_optimizer(cfg.actor_optimizer, self.actor)
        self.critic_optimizer = setup_optimizer(cfg.critic_optimizer, self.critic_1, self.critic_2)

def train_td3_step(td3: TD3):
        rb = td3.replay_buffer.get_shuffled(td3.cfg.algorithm.batch_size)
        rb_workspace = rb

        # compute q <- critic(s,a)
        td3.critic_1(rb_workspace, t=0)
        td3.critic_2(rb_workspace, t=0)
        q1 = rb_workspace["critic_1/q_value"]
        q2 = rb_workspace["critic_2/q_value"]
        reward, terminated = rb_workspace["env/reward", "env/terminated"]

        # a' = target_actor(s') + noise
        td3.target_actor(rb_workspace, t=1)
        target_action = rb_workspace["action"][1] 
        noise = torch.randn_like(target_action) * td3.cfg.algorithm.target_policy_noise
        noise = noise.clamp(
            -td3.cfg.algorithm.target_policy_noise_clip, 
            td3.cfg.algorithm.target_policy_noise_clip
        )
        low = torch.tensor(td3.train_env.action_space.low, device=target_action.device)
        high = torch.tensor(td3.train_env.action_space.high, device=target_action.device)
        smoothed_target_action = torch.max(torch.min(target_action + noise, high), low)

        rb_workspace.set("action", 1,smoothed_target_action)

        # compute t_q <- target_critic(s',a')
        with torch.no_grad():
            td3.target_critic_1(rb_workspace, t=1)
            td3.target_critic_2(rb_workspace, t=1)
            target_q1 = rb_workspace["target-critic_1/q_value"]
            target_q2 = rb_workspace["target-critic_2/q_value"]

        # y = r + gamma * (1 - done) * min(q1_t, q2_t)
        min_q = torch.min(target_q1, target_q2)
        must_bootstrap = ~terminated

        loss_q1 = compute_critic_loss(td3.cfg, reward, must_bootstrap, q1, min_q)
        loss_q2 = compute_critic_loss(td3.cfg, reward, must_bootstrap, q2, min_q)
        td3.logger.add_log("critic_1_loss", loss_q1, td3.nb_steps)
        td3.logger.add_log("critic_2_loss", loss_q2, td3.nb_steps)

        critic_loss = loss_q1 + loss_q2

        #Weights update on critics
        td3.critic_optimizer.zero_grad()
        critic_loss.backward()

        torch.nn.utils.clip_grad_norm_(
            td3.critic_1.parameters(), td3.cfg.algorithm.max_grad_norm
        )

        torch.nn.utils.clip_grad_norm_(
            td3.critic_2.parameters(), td3.cfg.algorithm.max_grad_norm
        )
        td3.critic_optimizer.step()

        # Si step % policy_delay == 0
        if td3.nb_steps % td3.cfg.algorithm.policy_delay == 0:
            # loss_actor = -critic_1(s, actor(s)).mean()
            td3.actor(rb_workspace, t=0)
            td3.critic_1(rb_workspace, t=0)
            q_values = rb_workspace["critic_1/q_value"]
            loss_actor = compute_actor_loss(q_values)

            #update of Actor's weights by backprop on critic
            td3.actor_optimizer.zero_grad()
            loss_actor.backward()
            torch.nn.utils.clip_grad_norm_(
                td3.actor.parameters(), td3.cfg.algorithm.max_grad_norm
            )
            td3.actor_optimizer.step()

            # soft_updates
            soft_update_params(td3.actor, td3.target_actor, td3.cfg.algorithm.tau_target)
            soft_update_params(td3.critic_1, td3.target_critic_1, td3.cfg.algorithm.tau_target)
            soft_update_params(td3.critic_2, td3.target_critic_2, td3.cfg.algorithm.tau_target)



In [88]:
from bbrl.workspace import Workspace
# for i in range(3, 10):
#     params["algorithm"]["seed"] = i
td3 = TD3(OmegaConf.create(params))
workspace = Workspace()
td3.train_agent(workspace, t=0, n_steps=1, stochastic=True)

for step in range(300000):
    td3.train_agent(workspace, t=1, n_steps=1, stochastic=True)
    transition = workspace.get_transitions()
    td3.replay_buffer.put(transition)

    td3.nb_steps += transition.batch_size()
    workspace.copy_n_last_steps(1)

    if td3.replay_buffer.size() > td3.cfg.algorithm.learning_starts:
        train_td3_step(td3)

    # evaluation + logging
    if td3.nb_steps % td3.cfg.algorithm.eval_interval == 0:
        td3.evaluate()


# run_td3(td3)
td3.visualize_best()

Video of best agent recorded in folder outputs\Swimmer-v5\td3-S3_20260211-131914\best_agent
moviepy is not installed, skipping video display
Video of best agent recorded in folder outputs\Swimmer-v5\td3-S4_20260211-140938\best_agent
moviepy is not installed, skipping video display
Video of best agent recorded in folder outputs\Swimmer-v5\td3-S5_20260211-145933\best_agent
moviepy is not installed, skipping video display
Video of best agent recorded in folder outputs\Swimmer-v5\td3-S6_20260211-154920\best_agent
moviepy is not installed, skipping video display
Video of best agent recorded in folder outputs\Swimmer-v5\td3-S7_20260211-163919\best_agent
moviepy is not installed, skipping video display
Video of best agent recorded in folder outputs\Swimmer-v5\td3-S8_20260211-172914\best_agent
moviepy is not installed, skipping video display
Video of best agent recorded in folder outputs\Swimmer-v5\td3-S9_20260211-181935\best_agent
moviepy is not installed, skipping video display


### Exercise 4:

You know have all the elements to study the impact of removing features from the environment
on the training performance, and the impact of temporally extending the agent in mitigating
partial observability, both with observation and with action extension.

In practice, you should produce the following learning curves:

- a learning curve of your algorithm on the standard Swimmer-v5 environment with full observability,
- two learning curves, one from removing $\dot{x}$ from Swimmer-v5 and the other from removing $\dot{\theta}$,
- one learning curve from removing both $\dot{x}$ and $\dot{\theta}$,
- the same four learning curves as above, but adding each of the temporal extension wrappers, separately or combined.

The way to combine these learning curves in different figures is open to you but should be carefully considered
depending on the conclusions you want to draw. Beware of drawing conclusions from insufficient statistics.

Discuss what you observe and conclude from this study.

In [ ]:
# [[STUDENT]]...

assert False, 'Not implemented yet'


# Lab report

Your report should contain:
- your source code (probably this notebook), do not forget to put your names on top of the notebook,
- in a separate pdf file with your names in the name of the file (name1_name2.pdf),  no longer than 6 pages:
    + a detailed enough description of all the choices you have made: the parameters you have set, the algorithms you have used, etc.,
    + the curves obtained when doing Exercise 3,
    + your conclusion from these experiments.

Beyond the elements required in this report, any additional study will be rewarded.
For instance, you can extend the temporal horizon for the state memory and or action sequences beyond 2,
and study the impact on learning performance and training time, etc.
A great achievement would be to perform a comparison with the approach based on an LSTM.